# 02 – SARIMA (Leitfaden Punkte 5–7)

Ziel: Ein saisonales ARIMA-Modell auf den Trainingsdaten identifizieren und anpassen, die 24 Testmonate prognostizieren und unter denselben Bedingungen wie Holt-Winters und die Seasonal-Naive-Baseline bewerten (gleicher Split, gleiche Metriken aus `src/evaluation.py`).

Regeln gegen Datenleckage:
- Die Abschnitte 1–6 verwenden **nur die Trainingsdaten**; die Testmenge wird dort gar nicht geladen (`train, _ = ...`).
- Das endgültige Modell wird in Abschnitt 6 **vor** der Prognose festgelegt.
- Die Testdaten werden erst in Abschnitt 7 einmalig verwendet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_umsatz
from src.evaluation import train_test_split
from src import arima
from src.plots import speichern

s = load_umsatz(str(ROOT / "data/raw/45212-0004_de.csv"))
train, _ = train_test_split(s)
print(f"Training: {train.index[0]:%m/%Y}–{train.index[-1]:%m/%Y} ({len(train)} Monate)")

## 1. Varianzstabilisierung (Box-Cox)

ARIMA setzt eine (schwach) stationäre Reihe voraus, also auch eine konstante Varianz (Vorlesung 3, Folie 5). Ist die Varianz offensichtlich nicht konstant, wird sie mit einer Box-Cox-Transformation stabilisiert (Vorlesung 2, Folien 32–34).

**Entscheidungsregel (vor der Analyse festgelegt):** Transformiert wird nur, wenn die Standardabweichung je Jahr deutlich mit dem Jahresmittelwert steigt (Vorgehen wie in der Beispiel-Hausarbeit, Kap. 3). Das per Software geschätzte λ wird ergänzend berichtet.

In [ ]:
tab = arima.jahres_streuung(train)
r = tab["mittelwert"].corr(tab["std"])
print(f"Vollständige Jahre: {tab.index[0]}–{tab.index[-1]} ({len(tab)} Jahre)")
print(f"Korrelation Jahresmittelwert – Standardabweichung: r = {r:.2f}")
tab.round(2)

In [ ]:
fig = arima.plot_jahres_streuung(tab)
speichern(fig, "08_jahresstreuung")

In [ ]:
lam, ki = arima.boxcox_lambda(train)
print(f"λ (Maximum Likelihood) = {lam:.2f}, 95%-Konfidenzintervall [{ki[0]:.2f}; {ki[1]:.2f}]")

**Ergebnis und Entscheidung:**
- Zwischen Jahresmittelwert und jährlicher Standardabweichung besteht praktisch kein Zusammenhang (r = 0,06).
- Die größte Streuung tritt bei niedrigem Niveau auf (1999: Mittelwert 87,06, Standardabweichung 7,53), die kleinste bei vergleichsweise hohem Niveau (2022: 95,50 bzw. 4,52).
- Die Varianz ist damit nicht offensichtlich niveauabhängig. Gemäß der festgelegten Regel wird **keine Box-Cox-Transformation** angewendet; modelliert wird die Originalreihe.
- Das per Maximum Likelihood geschätzte λ = −1,76 (95%-KI [−2,84; −0,70]) dient nicht als Entscheidungsgrundlage: Das Verfahren unterstellt unabhängige, identisch verteilte Beobachtungen, was bei einer Reihe mit Trend und Saison nicht erfüllt ist.
- Konsequenz: Die Prognosen müssen nicht zurücktransformiert werden, und AIC-Vergleiche erfolgen auf der Originalskala.

## 2. Differenzierung

Ziel ist eine stationäre Reihe: konstanter Mittelwert, konstante Varianz, Kovarianz nur vom zeitlichen Abstand abhängig (Vorlesung 3, Folie 5). Differenzen werden mit dem Backshift-Operator geschrieben: $(1-B)\,Y_t = Y_t - Y_{t-1}$ und saisonal $(1-B^{12})\,Y_t = Y_t - Y_{t-12}$ (Vorlesung 3, Folien 37 und 41).

**Vorgehen (vor der Analyse festgelegt):**
1. Zuerst saisonal differenzieren ($D = 1$), danach regulär ($d = 1$) nur, falls noch ein Trend erkennbar ist; höchstens $d + D = 2$.
2. Hauptkriterium wie in Vorlesung 3, Folie 38: Mittelwert und Varianz wirken stabil, und die ACF fällt relativ schnell ab. Leitsatz: „So wenig differenzieren wie nötig“, da Überdifferenzierung künstliche Autokorrelation erzeugt.
3. Ergänzend (nicht Teil der Vorlesung): ADF-Test ($H_0$: nicht stationär) und KPSS-Test ($H_0$: stationär). Stationarität wird gestützt, wenn ADF ablehnt und KPSS nicht ablehnt. KPSS-p-Werte werden nur im Bereich 0,01–0,10 ausgegeben.
4. Die zusätzlich regulär differenzierte Reihe wird zum Vergleich mitberechnet, um eine Überdifferenzierung zu erkennen.

In [ ]:
reihen = {
    "Original $Y_t$": train,
    "$(1-B^{12})\\,Y_t$": arima.differenzieren(train, D=1),
    "$(1-B)(1-B^{12})\\,Y_t$": arima.differenzieren(train, d=1, D=1),
}
diff_tab = arima.differenzen_tabelle(reihen)
diff_tab.round(3)

In [ ]:
fig = arima.plot_differenzen(reihen)
speichern(fig, "09_differenzen");

**Ergebnis und Entscheidung:**
- **Originalreihe:** nicht stationär. Die ACF ist bei den saisonalen Lags hoch (Lag 12: 0,868) und fällt nur langsam ab; ADF lehnt nicht ab (p = 0,252), KPSS lehnt Stationarität ab (p ≤ 0,01).
- **Nach saisonaler Differenz $(1-B^{12})$:** Die Varianz sinkt von 52,59 auf 11,09, die Autokorrelation bei Lag 12 verschwindet (−0,005). Die Reihe schwankt um einen konstanten Mittelwert; die ACF liegt nach den ersten Lags überwiegend im Konfidenzband und zeigt keinen langsamen, linearen Abfall wie ein Random Walk (Vorlesung 3, Folie 12). ADF lehnt ab (p = 0,006), KPSS lehnt nicht ab (p ≥ 0,10): beide Tests stützen Stationarität.
- **Zusätzliche reguläre Differenz:** Die Autokorrelation bei Lag 1 wird stark negativ (−0,500) und die Varianz steigt auf 15,68. Das sind Anzeichen einer Überdifferenzierung (Vorlesung 3, Folie 38). Die Tests erkennen dies nicht (ADF p = 0,000), daher sind Grafik und ACF das Hauptkriterium.
- **Entscheidung:** $D = 1$, $d = 0$. Da $d + D = 1$, wird ein konstanter Term in das Modell aufgenommen und seine Signifikanz in Abschnitt 5 geprüft.
- Die verbleibende kurzfristige Autokorrelation der saisonal differenzierten Reihe wird in Abschnitt 3 über AR- und MA-Terme modelliert.